In [8]:
import pandas as pd

# Read Data Raw
df_raw = pd.read_csv('Global_Education.csv',  encoding='latin1')

# TRANSFORM (Cleaning dan Modelling)
df_cleaned = df_raw.drop_duplicates().dropna()

# --- TABEL DIMENSI ---
# Dimensi Country
df_cleaned['country_id'] = range(1, len(df_cleaned) + 1)

dim_country = df_cleaned[
    ['country_id', 'Countries and areas', 'Latitude ', 'Longitude']
].drop_duplicates().reset_index(drop=True)

# Rename kolom agar aman masuk ke database
dim_country = dim_country.rename(columns={
    'Countries and areas': 'country_name', 
    'Latitude ': 'latitude',
    'Longitude': 'longitude'
})

# --- TABEL FAKTA ---
fact_global_education_metrics = df_cleaned.copy()

fact_global_education_metrics = fact_global_education_metrics.drop(
  columns=[
    'Countries and areas', 
    'Latitude ', 
    'Longitude'
  ])

fact_global_education_metrics.columns = fact_global_education_metrics.columns.str.lower().str.replace(' ', '_')

fact_global_education_metrics['fact_id'] = range(1, len(fact_global_education_metrics) + 1)
fact_global_education_metrics = fact_global_education_metrics[['fact_id'] + [c for c in fact_global_education_metrics.columns if c != 'fact_id']]


# CEK DATA
print("Kolom Dimensi:", dim_country.columns.tolist())
print("\nKolom Fakta:", fact_global_education_metrics.columns.tolist())

fact_global_education_metrics.head()


# --- LOAD KE SUPABASE
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
engine = create_engine(DATABASE_URL)

print("Memulai proses load data...")
dim_country.to_sql(
    'dim_country',
    con=engine,
    if_exists='replace',
    index=False
)

fact_global_education_metrics.to_sql(
    'fact_global_education_metrics',
    con=engine,
    if_exists='replace',
    index=False
)
print("FASE LOAD SELESAI 100%!...")

Kolom Dimensi: ['country_id', 'country_name', 'latitude', 'longitude']

Kolom Fakta: ['fact_id', 'oosr_pre0primary_age_male', 'oosr_pre0primary_age_female', 'oosr_primary_age_male', 'oosr_primary_age_female', 'oosr_lower_secondary_age_male', 'oosr_lower_secondary_age_female', 'oosr_upper_secondary_age_male', 'oosr_upper_secondary_age_female', 'completion_rate_primary_male', 'completion_rate_primary_female', 'completion_rate_lower_secondary_male', 'completion_rate_lower_secondary_female', 'completion_rate_upper_secondary_male', 'completion_rate_upper_secondary_female', 'grade_2_3_proficiency_reading', 'grade_2_3_proficiency_math', 'primary_end_proficiency_reading', 'primary_end_proficiency_math', 'lower_secondary_end_proficiency_reading', 'lower_secondary_end_proficiency_math', 'youth_15_24_literacy_rate_male', 'youth_15_24_literacy_rate_female', 'birth_rate', 'gross_primary_education_enrollment', 'gross_tertiary_education_enrollment', 'unemployment_rate', 'country_id']
Memulai proses l